In [ ]:
"""
kaggle_zenodo_download.py
─────────────────────────────────────────────────────────────────
Chạy trong Kaggle Notebook để tải Forest Inspection Dataset
(Zenodo #15511426) thẳng vào /kaggle/working/ — server-to-server.

Sau khi chạy xong: Save Version → "Save & Run All" → output files
sẽ xuất hiện ở tab "Output" → bấm "New Dataset" để publish.
─────────────────────────────────────────────────────────────────
"""

import logging
import sys
import time
import hashlib
import requests
from pathlib import Path
from tqdm import tqdm

# ─── Config ───────────────────────────────────────────────────────────────────
ZENODO_RECORD = "15511426"
OUTPUT_DIR    = Path("/kaggle/working/UAV_Forest_Overcast")

FILES = [
    # (filename,  size_GB,  md5)
    # ("README.md", 0.000003, "e18b2c85d02098cee955957668b5f24a"),
    # ("seq1.zip",  3.7,      "b05b6a9718bab34fd88f99dbc00762ba"),
    # ("seq2.zip",  4.8,      "23843f0d274b28faf742afd75bff30a0"),
    # ("seq3.zip",  4.6,      "86987b2c013f2f92d9dc0b69feb21aed"),
    # ("seq4.zip",  3.2,      "dec96c8c23469240e33d08887c813a01"),
    ("seq5.zip",  4.8,      "701c9d7d47b85d4b82561cb55cb7a1f3"),
    ("seq6.zip",  4.7,      "62550cb3231677beb94c07bb09788147"),
    ("seq7.zip",  2.8,      "a529e8902e4ee0868c87e2aec9558f88"),
    ("seq8.zip",  4.9,      "a718c43245e5be962afceb5e0ec2800c"),
    ("seq9.zip",  4.6,      "8daf9b8ea8c6b7ee641c677b8f865e44"),
]

# ─── Logging ──────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger(__name__)

# ─── Helpers ──────────────────────────────────────────────────────────────────
def md5_file(path: Path) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


HEADERS = {"User-Agent": "Mozilla/5.0 zenodo-kaggle-downloader/1.0"}


def download_file(filename: str, size_gb: float, expected_md5: str) -> str:
    """
    Download 1 file từ Zenodo bằng requests + tqdm progress bar.
    Trả về: 'success' | 'skipped' | 'failed'
    """
    dest = OUTPUT_DIR / filename
    url  = f"https://zenodo.org/records/{ZENODO_RECORD}/files/{filename}?download=1"

    # ── Skip nếu file đã tồn tại và đúng kích thước ──────────────────────────
    if dest.exists() and dest.stat().st_size > 1024:
        logger.info(f"  [SKIP] Đã tồn tại: {filename}  ({dest.stat().st_size / 1e9:.2f} GB)")
        return "skipped"

    logger.info(f"  ↓ Bắt đầu tải: {filename}  (~{size_gb} GB)")
    t0 = time.time()

    try:
        resp = requests.get(url, headers=HEADERS, stream=True, timeout=120)
        resp.raise_for_status()

        total_bytes = int(resp.headers.get("Content-Length", 0))
        chunk_size  = 1024 * 1024  # 1 MB

        with open(dest, "wb") as f, tqdm(
            total=total_bytes or None,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc=f"  {filename}",
            ncols=80,
            colour="green",
            leave=True,
        ) as bar:
            for chunk in resp.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))

    except Exception as e:
        logger.error(f"  ✗ Lỗi tải {filename}: {e}")
        if dest.exists():
            dest.unlink()   # xóa file lỗi/rỗng
        return "failed"

    elapsed = time.time() - t0
    minutes = int(elapsed // 60)
    seconds = int(elapsed % 60)
    actual_gb = dest.stat().st_size / 1e9
    logger.info(f"  ✓ Đã tải: {filename}  ({actual_gb:.2f} GB | {minutes}m{seconds:02d}s)")

    # ── Xác minh MD5 (bỏ qua README do size nhỏ không cần thiết) ─────────────
    if expected_md5 and not filename.endswith(".md"):
        logger.info(f"    Đang kiểm tra MD5...")
        actual_md5 = md5_file(dest)
        if actual_md5 == expected_md5:
            logger.info(f"    ✓ MD5 khớp: {actual_md5}")
        else:
            logger.warning(f"    ⚠ MD5 KHÔNG KHỚP!")
            logger.warning(f"      Mong đợi : {expected_md5}")
            logger.warning(f"      Thực tế  : {actual_md5}")

    return "success"


# ─── Main ─────────────────────────────────────────────────────────────────────
def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    total_gb = sum(s for _, s, _ in FILES)
    logger.info("=" * 65)
    logger.info("Forest Inspection Dataset - Sunny sequences (Zenodo #15511426)")
    logger.info(f"Tổng: {len(FILES)} files  |  ~{total_gb:.1f} GB")
    logger.info(f"Lưu vào: {OUTPUT_DIR}")
    logger.info("=" * 65)

    results = {"success": 0, "skipped": 0, "failed": 0}
    total_start = time.time()

    for idx, (filename, size_gb, md5) in enumerate(FILES, start=1):
        logger.info(f"\n[{idx}/{len(FILES)}]  {filename}")
        status = download_file(filename, size_gb, md5)
        results[status] += 1

    # ── Tóm tắt ──────────────────────────────────────────────────────────────
    total_elapsed = time.time() - total_start
    total_min     = int(total_elapsed // 60)

    logger.info("\n" + "=" * 65)
    logger.info("KẾT QUẢ:")
    logger.info(f"  ✓ Thành công : {results['success']}")
    logger.info(f"  → Bỏ qua     : {results['skipped']}")
    logger.info(f"  ✗ Thất bại   : {results['failed']}")
    logger.info(f"  ⏱ Tổng thời gian: {total_min} phút")
    logger.info(f"  📂 Output: {OUTPUT_DIR}")
    logger.info("=" * 65)

    # ── Liệt kê file đã tải ──────────────────────────────────────────────────
    logger.info("\nFiles trong output:")
    for f in sorted(OUTPUT_DIR.iterdir()):
        size_gb = f.stat().st_size / 1e9
        logger.info(f"  {f.name:<15}  {size_gb:.3f} GB")


if __name__ == "__main__":
    main()